# 01 — Data Understanding & Quality Checks

**Goal:** Validate the four SaaS datasets before analysis.

Checks include:
- shape and schema
- data types
- null values
- duplicate rows and primary keys
- referential integrity
- logically invalid date sequences


In [2]:
import numpy as np
import pandas as pd

users = pd.read_csv("users.csv")
events = pd.read_csv("events.csv")
sessions = pd.read_csv("sessions.csv")
subscriptions = pd.read_csv("subscriptions.csv")

## 1. Dataset size


In [12]:
print("users:", users.shape)
print("events:", events.shape)
print("sessions:", sessions.shape)
print("subscriptions:", subscriptions.shape)

users: (10000, 6)
events: (121480, 6)
sessions: (66195, 5)
subscriptions: (3097, 7)


## 2. Initial inspection


In [13]:
display(users.head())
display(events.head())
display(sessions.head())
display(subscriptions.head())

,user_id,signup_date,country,acquisition_channel,device,company_size
0,U00001,2025-04-13,Germany,Social,Desktop,Large
1,U00002,2025-06-29,UAE,Organic,Desktop,Medium
2,U00003,2025-04-03,India,Organic,Desktop,Medium
3,U00004,2025-01-15,India,Referral,Mobile,Small
4,U00005,2025-04-17,India,Organic,Desktop,Small


,event_id,user_id,event_timestamp,event_name,feature_name,session_id
0,E00000001,U07970,2025-01-01 01:00:00,project_created,NaN,SS0052849
1,E00000002,U03298,2025-01-01 01:00:00,project_created,NaN,SS0022199
2,E00000003,U06077,2025-01-01 01:00:00,project_created,NaN,SS0040628
3,E00000004,U09633,2025-01-01 01:00:00,project_created,NaN,SS0063806
4,E00000005,U06661,2025-01-01 02:00:00,project_created,NaN,SS0044578


,session_id,user_id,session_start,session_end,device
0,SS0000001,U00001,2025-05-27 15:32:00,2025-05-27 15:53:00,Desktop
1,SS0000002,U00001,2025-07-06 09:48:00,2025-07-06 10:07:00,Desktop
2,SS0000003,U00001,2025-06-16 11:58:00,2025-06-16 12:06:00,Desktop
3,SS0000004,U00001,2025-05-30 16:06:00,2025-05-30 16:25:00,Desktop
4,SS0000005,U00001,2025-05-20 14:48:00,2025-05-20 15:44:00,Desktop


,subscription_id,user_id,trial_start_date,subscription_date,plan,monthly_price,cancel_date
0,S00001,U00001,2025-04-22,2025-05-04,Pro,49.0,NaT
1,S00002,U00002,2025-07-08,NaT,NaN,NaN,NaT
2,S00003,U00004,2025-01-21,NaT,NaN,NaN,NaT
3,S00004,U00011,2025-04-08,2025-04-21,Pro,49.0,NaT
4,S00005,U00012,2025-04-30,2025-05-06,Pro,49.0,NaT


In [14]:
users.info()
events.info()
sessions.info()
subscriptions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   user_id              10000 non-null  object        
 1   signup_date          10000 non-null  datetime64[ns]
 2   country              10000 non-null  object        
 3   acquisition_channel  10000 non-null  object        
 4   device               10000 non-null  object        
 5   company_size         10000 non-null  object        
dtypes: datetime64[ns](1), object(5)
memory usage: 468.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121480 entries, 0 to 121479
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   event_id         121480 non-null  object        
 1   user_id          121480 non-null  object        
 2   event_timestamp  121480 non-null  datetime64[

## 3. Convert date columns


In [15]:
users["signup_date"] = pd.to_datetime(users["signup_date"], errors="coerce")
events["event_timestamp"] = pd.to_datetime(events["event_timestamp"], errors="coerce")
sessions["session_start"] = pd.to_datetime(sessions["session_start"], errors="coerce")
sessions["session_end"] = pd.to_datetime(sessions["session_end"], errors="coerce")

for col in ["trial_start_date", "subscription_date", "cancel_date"]:
    subscriptions[col] = pd.to_datetime(subscriptions[col], errors="coerce")


## 4. Null-value checks


In [16]:
print("USERS")
display(users.isnull().sum())

print("EVENTS")
display(events.isnull().sum())

print("SESSIONS")
display(sessions.isnull().sum())

print("SUBSCRIPTIONS")
display(subscriptions.isnull().sum())


USERS


user_id                0
signup_date            0
country                0
acquisition_channel    0
device                 0
company_size           0
dtype: int64

EVENTS


event_id               0
user_id                0
event_timestamp        0
event_name             0
feature_name       97961
session_id             0
dtype: int64

SESSIONS


session_id       0
user_id          0
session_start    0
session_end      0
device           0
dtype: int64

SUBSCRIPTIONS


subscription_id         0
user_id                 0
trial_start_date        0
subscription_date    1125
plan                 1125
monthly_price        1125
cancel_date          2830
dtype: int64

**Business note:** Null subscription fields are not automatically data-quality issues.
They can represent valid states such as trial users who did not convert or active
subscribers who have not cancelled.


## 5. Duplicate checks


In [17]:
print("Duplicate rows")
print("users:", users.duplicated().sum())
print("events:", events.duplicated().sum())
print("sessions:", sessions.duplicated().sum())
print("subscriptions:", subscriptions.duplicated().sum())

print("\nDuplicate primary keys")
print("users.user_id:", users["user_id"].duplicated().sum())
print("events.event_id:", events["event_id"].duplicated().sum())
print("sessions.session_id:", sessions["session_id"].duplicated().sum())
print("subscriptions.subscription_id:", subscriptions["subscription_id"].duplicated().sum())


Duplicate rows
users: 0
events: 0
sessions: 0
subscriptions: 0

Duplicate primary keys
users.user_id: 0
events.event_id: 0
sessions.session_id: 0
subscriptions.subscription_id: 0


## 6. Referential integrity


In [18]:
print("Orphan events:", (~events["user_id"].isin(users["user_id"])).sum())
print("Orphan sessions:", (~sessions["user_id"].isin(users["user_id"])).sum())
print("Orphan subscriptions:", (~subscriptions["user_id"].isin(users["user_id"])).sum())


Orphan events: 0
Orphan sessions: 0
Orphan subscriptions: 0


## 7. Business-validity checks


In [19]:
invalid_sessions = sessions["session_end"] < sessions["session_start"]

invalid_subscription_dates = (
    subscriptions["subscription_date"].notna()
    & subscriptions["trial_start_date"].notna()
    & (subscriptions["subscription_date"] < subscriptions["trial_start_date"])
)

invalid_cancel_dates = (
    subscriptions["cancel_date"].notna()
    & subscriptions["subscription_date"].notna()
    & (subscriptions["cancel_date"] < subscriptions["subscription_date"])
)

event_signup_check = events.merge(
    users[["user_id", "signup_date"]],
    on="user_id",
    how="left"
)

invalid_event_dates = (
    event_signup_check["event_timestamp"] < event_signup_check["signup_date"]
)

print("Sessions ending before start:", invalid_sessions.sum())
print("Subscriptions before trial:", invalid_subscription_dates.sum())
print("Cancellations before subscription:", invalid_cancel_dates.sum())
print("Events before signup:", invalid_event_dates.sum())


Sessions ending before start: 0
Subscriptions before trial: 0
Cancellations before subscription: 0
Events before signup: 0


## Data-quality conclusion

Document the final result here after running the notebook.

Suggested wording if the checks remain clean:

> No duplicate records, duplicate primary keys, orphan foreign keys, or logically
> invalid date sequences were identified. Business-valid null subscription states
> were retained.
